# Unemployment Analysis in India Using Python
## Unemployment Trends, COVID-19 Disruption, and Regional Analysis (May 2019 – June 2020)

**Author:** Data Science & Analytics Team  
**Technology Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Streamlit  
**Dataset:** Unemployment in India Dataset (740 clean observations across 28 States & UTs)  

---

### Executive Summary & Objectives
Unemployment is a critical macroeconomic indicator reflecting economic health, labor market dynamics, and social stability. This project performs an end-to-end data analysis of monthly unemployment rates, employed population, and labor participation rates in India between May 2019 and June 2020. A primary focus is evaluating the economic shock of the **COVID-19 nationwide lockdown** initiated in March 2020.

**Key Objectives:**
1. Load, inspect, and clean raw employment data.
2. Perform Exploratory Data Analysis (EDA) and compute overall summary statistics.
3. Analyze unemployment rate trends over time with moving averages.
4. Quantify the impact of COVID-19 by comparing Pre-COVID (May 2019–Feb 2020) vs COVID-19 (Mar 2020–Jun 2020) periods.
5. Evaluate regional and state-wise variations across 28 Indian States/UTs.
6. Examine Rural vs Urban labor market disparities.
7. Analyze correlation patterns between Unemployment Rate, Employed Population, and Labour Participation Rate.
8. Identify extreme statistical outliers and volatility spikes.

In [ ]:
# Step 1: Environment Setup and Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Set global plotting aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("Libraries loaded successfully!")

In [ ]:
# Step 2: Data Loading & Preprocessing
dataset_path = '../data/unemployment_dataset.csv'
df_raw = pd.read_csv(dataset_path)

print(f"Raw Dataset Shape: {df_raw.shape}")
print("\nRaw Column Names:", repr(df_raw.columns.tolist()))

# Clean column names (strip leading/trailing whitespace)
df_raw.columns = df_raw.columns.str.strip()

# Standardize column dictionary mapping
df = df_raw.rename(columns={
    'Region': 'Region',
    'Date': 'Date',
    'Frequency': 'Frequency',
    'Estimated Unemployment Rate (%)': 'Unemployment_Rate',
    'Estimated Employed': 'Employed',
    'Estimated Labour Participation Rate (%)': 'Labour_Participation_Rate',
    'Area': 'Area'
}).copy()

# Drop completely empty rows and invalid rows
df = df.dropna(subset=['Region', 'Date']).reset_index(drop=True)

# Clean string columns
for col in ['Region', 'Frequency', 'Area']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Convert Date to datetime format
df['Date'] = pd.to_datetime(df['Date'].str.strip(), format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['Date'])

# Convert numeric columns
num_cols = ['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=num_cols)

# Feature engineering
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['COVID_Period'] = np.where(df['Date'] < '2020-03-01', 'Pre-COVID', 'COVID-19 Period')

print(f"Cleaned Dataset Shape: {df.shape}")
display(df.head())

### Step 3: Exploratory Data Analysis & Overall Statistics
We calculate the statistical metrics (Mean, Median, Standard Deviation, Min, Max, Quantiles) for Unemployment Rate, Employed Population, and Labour Participation Rate.

In [ ]:
# Statistical Summary Table
summary_stats = df[['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']].describe().T
summary_stats['median'] = df[['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']].median()
summary_stats['iqr'] = summary_stats['75%'] - summary_stats['25%']
display(summary_stats[['mean', 'median', 'std', 'min', '25%', '75%', 'iqr', 'max']])

In [ ]:
# Plot 1: Unemployment Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['Unemployment_Rate'], kde=True, color='#1f77b4', bins=30, edgecolor='black')
plt.axvline(df['Unemployment_Rate'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean ({df['Unemployment_Rate'].mean():.2f}%)")
plt.axvline(df['Unemployment_Rate'].median(), color='green', linestyle='-', linewidth=2, label=f"Median ({df['Unemployment_Rate'].median():.2f}%)")
plt.title('Distribution of Estimated Unemployment Rate (%) in India', fontweight='bold')
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

### Step 4: Overall Unemployment Trend & COVID-19 Impact
We compute the monthly average unemployment rate across all regions and compute a 3-month moving average to analyze the macro trend.

In [ ]:
# Plot 2: Time Series Line Chart
monthly_avg = df.groupby('Date')['Unemployment_Rate'].mean().reset_index()
monthly_avg['3M_MA'] = monthly_avg['Unemployment_Rate'].rolling(window=3, min_periods=1).mean()

plt.figure(figsize=(12, 6))
plt.plot(monthly_avg['Date'], monthly_avg['Unemployment_Rate'], marker='o', linewidth=2.5, color='#d62728', label='Monthly Mean Unemployment Rate')
plt.plot(monthly_avg['Date'], monthly_avg['3M_MA'], linestyle='--', linewidth=2, color='#1f77b4', label='3-Month Moving Average')
plt.axvspan(pd.to_datetime('2020-03-01'), pd.to_datetime('2020-06-30'), color='#ffbbbb', alpha=0.4, label='COVID-19 Lockdown Peak')
plt.title('Overall Unemployment Rate Trend in India (May 2019 – June 2020)', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Unemployment Rate (%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Quantifying Pre-COVID vs COVID-19 Disruption
covid_summary = df.groupby('COVID_Period')[['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']].agg(
    Mean_Unemployment=('Unemployment_Rate', 'mean'),
    Median_Unemployment=('Unemployment_Rate', 'median'),
    Max_Unemployment=('Unemployment_Rate', 'max'),
    Mean_Employed=('Employed', 'mean'),
    Mean_Labour_Participation=('Labour_Participation_Rate', 'mean')
).reindex(['Pre-COVID', 'COVID-19 Period'])

display(covid_summary)

### Step 5: Regional & State-level Analysis
We aggregate unemployment metrics by state to identify regions with the highest and lowest baseline unemployment.

In [ ]:
# Regional Rankings
reg_stats = df.groupby('Region')['Unemployment_Rate'].agg(['mean', 'max', 'min', 'std']).sort_values(by='mean', ascending=False)
print("Top 5 Highest Unemployment States:")
display(reg_stats.head(5))

print("\nTop 5 Lowest Unemployment States:")
display(reg_stats.tail(5))

In [ ]:
# Plot 3: Top 10 Highest Unemployment States Bar Chart
top10 = reg_stats.head(10).reset_index()
plt.figure(figsize=(10, 5))
sns.barplot(data=top10, x='Region', y='mean', palette='Reds_r', edgecolor='black')
plt.title('Top 10 States with Highest Mean Unemployment Rate (%)', fontweight='bold')
plt.xlabel('State / Region')
plt.ylabel('Mean Unemployment Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Step 6: Rural vs Urban Breakdown & Correlation Analysis
We evaluate structural differences between Rural and Urban areas and construct the correlation matrix between key economic indicators.

In [ ]:
# Rural vs Urban Comparison
area_stats = df.groupby('Area')[['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']].mean()
display(area_stats)

# Correlation Heatmap
plt.figure(figsize=(7, 5))
corr = df[['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='Blues', vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix of Economic Metrics', fontweight='bold')
plt.tight_layout()
plt.show()

### Step 7: Key Findings & Summary
1. **Overall Mean Unemployment Rate:** 11.79% across all observations (May 2019 – June 2020).
2. **COVID-19 Shock:** Mean unemployment surged from **9.51%** pre-COVID to **17.77%** during COVID (an **86.91%** increase), peaking in April (23.64%) and May (24.88%) 2020.
3. **Employment Loss:** Mean employed population dropped from **7.47 Million** to **6.52 Million** per region.
4. **Urban Disparity:** Urban areas recorded a higher mean unemployment rate (**13.17%**) than Rural areas (**10.32%**).
5. **Extreme Spikes:** Puducherry (76.74%), Jharkhand (70.17%), Bihar (58.77%), Tamil Nadu (53.19%), and Himachal Pradesh (50.00%) experienced severe spikes during April–May 2020.